In [3]:
from pyspark.sql import SparkSession

try:
    spark.stop()
    print("SparkContext anterior detenido")
except:
    print("No había SparkContext activo")

spark = SparkSession.builder \
    .appName("PruebaConexiones") \
    .master("local[*]") \
    .config("spark.mongodb.read.connection.uri", "mongodb://mongo:27017/smartcity.lecturas") \
    .config("spark.mongodb.write.connection.uri", "mongodb://mongo:27017/smartcity.resultados") \
    .config("spark.hadoop.fs.defaultFS", "hdfs://namenode:9000") \
    .getOrCreate()

print("Spark listo:", spark.version)
print("Master:", spark.sparkContext.master)
print("App Name:", spark.sparkContext.appName)
print("UI Web:", spark.sparkContext.uiWebUrl)

SparkContext anterior detenido
Spark listo: 3.5.0
Master: local[*]
App Name: PruebaConexiones
UI Web: http://2175cf5ae36d:4040


In [6]:
import pandas as pd
from pymongo import MongoClient
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, IntegerType, TimestampType

print("=== PRUEBA 1: CONEXIÓN A MONGODB (vía pymongo) ===")
print("Leyendo de: mongodb://mongo:27017/smartcity.lecturas")

# Conectar a Mongo con pymongo
client = MongoClient("mongodb://mongo:27017")
coleccion = client["smartcity"]["lecturas"]

# Leer los primeros 10000 documentos
documentos = list(coleccion.find({}, {"_id": 0}).limit(10000))
print(f"✅ MongoDB: {len(documentos)} documentos leídos con pymongo")

# Convertir a pandas DataFrame
df_pandas = pd.DataFrame(documentos)

# Aplanar columnas anidadas
df_pandas["sensor_id"] = df_pandas["sensor"].apply(lambda x: x.get("id") if isinstance(x, dict) else None)
df_pandas["sensor_zona"] = df_pandas["sensor"].apply(lambda x: x.get("zona") if isinstance(x, dict) else None)
df_pandas["sensor_tipo"] = df_pandas["sensor"].apply(lambda x: x.get("tipo") if isinstance(x, dict) else None)
df_pandas["pm25"] = df_pandas["medicion"].apply(lambda x: x.get("pm25") if isinstance(x, dict) else None)
df_pandas["carros"] = df_pandas["medicion"].apply(lambda x: x.get("carros") if isinstance(x, dict) else None)

# Seleccionar columnas
df_limpio = df_pandas[["sensor_id", "sensor_zona", "sensor_tipo", "pm25", "carros", "timestamp"]].copy()

# Convertir tipos ANTES de pasar a Spark
df_limpio["pm25"] = pd.to_numeric(df_limpio["pm25"], errors="coerce")
df_limpio["carros"] = pd.to_numeric(df_limpio["carros"], errors="coerce")
df_limpio["sensor_id"] = df_limpio["sensor_id"].astype(str)
df_limpio["sensor_zona"] = df_limpio["sensor_zona"].astype(str)
df_limpio["sensor_tipo"] = df_limpio["sensor_tipo"].astype(str)
df_limpio["timestamp"] = pd.to_datetime(df_limpio["timestamp"], errors="coerce")

# Esquema explícito para Spark
esquema = StructType([
    StructField("sensor_id", StringType(), True),
    StructField("sensor_zona", StringType(), True),
    StructField("sensor_tipo", StringType(), True),
    StructField("pm25", DoubleType(), True),
    StructField("carros", IntegerType(), True),
    StructField("timestamp", TimestampType(), True),
])

# Convertir a Spark DataFrame con esquema explícito
df_mongo = spark.createDataFrame(df_limpio, schema=esquema)

print(f"✅ Spark DataFrame creado con {df_mongo.count()} filas")
df_mongo.printSchema()
df_mongo.show(5, truncate=False)

=== PRUEBA 1: CONEXIÓN A MONGODB (vía pymongo) ===
Leyendo de: mongodb://mongo:27017/smartcity.lecturas
✅ MongoDB: 10000 documentos leídos con pymongo
✅ Spark DataFrame creado con 10000 filas
root
 |-- sensor_id: string (nullable = true)
 |-- sensor_zona: string (nullable = true)
 |-- sensor_tipo: string (nullable = true)
 |-- pm25: double (nullable = true)
 |-- carros: integer (nullable = true)
 |-- timestamp: timestamp (nullable = true)

+---------+------------------+-----------+----+------+-------------------+
|sensor_id|sensor_zona       |sensor_tipo|pm25|carros|timestamp          |
+---------+------------------+-----------+----+------+-------------------+
|INT_001  |Financial District|trafico    |NaN |162   |2026-01-01 00:00:00|
|INT_001  |Financial District|trafico    |NaN |150   |2026-01-01 01:00:00|
|INT_001  |Financial District|trafico    |NaN |186   |2026-01-01 02:00:00|
|INT_001  |Financial District|trafico    |NaN |162   |2026-01-01 03:00:00|
|INT_001  |Financial District|t

In [7]:
print("=== PRUEBA 2: CONEXIÓN A POSTGRESQL ===")
print("Leyendo de: jdbc:postgresql://postgres:5432/smartcity")

df_postgres = spark.read.format("jdbc") \
    .option("url", "jdbc:postgresql://postgres:5432/smartcity") \
    .option("dbtable", "medicion") \
    .option("user", "admin") \
    .option("password", "admin123") \
    .option("driver", "org.postgresql.Driver") \
    .load()

print(f"✅ PostgreSQL: {df_postgres.count()} filas leídas de 'medicion'")
print(f"Columnas: {df_postgres.columns}")
df_postgres.printSchema()
df_postgres.show(3, truncate=False)

=== PRUEBA 2: CONEXIÓN A POSTGRESQL ===
Leyendo de: jdbc:postgresql://postgres:5432/smartcity
✅ PostgreSQL: 1062858 filas leídas de 'medicion'
Columnas: ['id_medicion', 'id_sensor', 'pm25', 'pm10', 'no2', 'o3', 'co', 'carros', 'temperatura', 'aqi', 'congestion_score', 'emission_estimate', 'fuel_waste_estimate', 'accidente', 'es_fin_semana', 'fecha']
root
 |-- id_medicion: integer (nullable = true)
 |-- id_sensor: integer (nullable = true)
 |-- pm25: double (nullable = true)
 |-- pm10: double (nullable = true)
 |-- no2: double (nullable = true)
 |-- o3: double (nullable = true)
 |-- co: double (nullable = true)
 |-- carros: integer (nullable = true)
 |-- temperatura: double (nullable = true)
 |-- aqi: integer (nullable = true)
 |-- congestion_score: double (nullable = true)
 |-- emission_estimate: double (nullable = true)
 |-- fuel_waste_estimate: double (nullable = true)
 |-- accidente: boolean (nullable = true)
 |-- es_fin_semana: boolean (nullable = true)
 |-- fecha: timestamp (nulla

In [8]:
print("=== PRUEBA 3: CONEXIÓN A HDFS ===")
print("Leyendo de: hdfs://namenode:9000/historico/datos_limpios.csv")

df_hdfs = spark.read.csv(
    "hdfs://namenode:9000/historico/datos_limpios.csv",
    header=True,
    inferSchema=True
)

total_hdfs = df_hdfs.count()
print(f"✅ HDFS: {total_hdfs} filas leídas del CSV")
print(f"Columnas: {len(df_hdfs.columns)} columnas")
df_hdfs.printSchema()
df_hdfs.show(3, truncate=False)

=== PRUEBA 3: CONEXIÓN A HDFS ===
Leyendo de: hdfs://namenode:9000/historico/datos_limpios.csv
✅ HDFS: 1062858 filas leídas del CSV
Columnas: 18 columnas
root
 |-- sensor_nombre: string (nullable = true)
 |-- zona: string (nullable = true)
 |-- fecha: timestamp (nullable = true)
 |-- pm25: double (nullable = true)
 |-- pm10: double (nullable = true)
 |-- no2: double (nullable = true)
 |-- o3: double (nullable = true)
 |-- co: double (nullable = true)
 |-- carros: double (nullable = true)
 |-- aqi: double (nullable = true)
 |-- temperatura: double (nullable = true)
 |-- congestion_score: double (nullable = true)
 |-- emission_estimate: double (nullable = true)
 |-- fuel_waste_estimate: double (nullable = true)
 |-- accidente: integer (nullable = true)
 |-- es_fin_semana: integer (nullable = true)
 |-- latitud: double (nullable = true)
 |-- longitud: double (nullable = true)

+-------------+------------------+-------------------+----+----+----+----+----+------+----+-----------+----------

In [9]:
print("=" * 60)
print("=== RESUMEN DE CONEXIONES ===")
print("=" * 60)
print(f"✅ MongoDB    → {df_mongo.count():,} documentos")
print(f"✅ PostgreSQL → {df_postgres.count():,} filas")
print(f"✅ HDFS       → {total_hdfs:,} filas")
print("=" * 60)

print(f"\nParticiones del DataFrame HDFS: {df_hdfs.rdd.getNumPartitions()}")
print("(Si es > 1, Spark está distribuyendo los datos)")

print("\n¡Las 3 conexiones funcionan correctamente!")

=== RESUMEN DE CONEXIONES ===
✅ MongoDB    → 10,000 documentos
✅ PostgreSQL → 1,062,858 filas
✅ HDFS       → 1,062,858 filas

Particiones del DataFrame HDFS: 20
(Si es > 1, Spark está distribuyendo los datos)

¡Las 3 conexiones funcionan correctamente!


In [1]:
import os

jars = ",".join([
    "/usr/local/spark-3.5.0-bin-hadoop3/jars/postgresql-42.7.3.jar",
    "/usr/local/spark-3.5.0-bin-hadoop3/jars/mongo-spark-connector_2.12-10.3.0.jar",
    "/usr/local/spark-3.5.0-bin-hadoop3/jars/mongodb-driver-sync-4.11.1.jar",
    "/usr/local/spark-3.5.0-bin-hadoop3/jars/mongodb-driver-core-4.11.1.jar",
    "/usr/local/spark-3.5.0-bin-hadoop3/jars/bson-4.11.1.jar"
])

os.environ['PYSPARK_SUBMIT_ARGS'] = f'--jars {jars} pyspark-shell'

print("PYSPARK_SUBMIT_ARGS configurado:")
print(os.environ['PYSPARK_SUBMIT_ARGS'])

PYSPARK_SUBMIT_ARGS configurado:
--jars /usr/local/spark-3.5.0-bin-hadoop3/jars/postgresql-42.7.3.jar,/usr/local/spark-3.5.0-bin-hadoop3/jars/mongo-spark-connector_2.12-10.3.0.jar,/usr/local/spark-3.5.0-bin-hadoop3/jars/mongodb-driver-sync-4.11.1.jar,/usr/local/spark-3.5.0-bin-hadoop3/jars/mongodb-driver-core-4.11.1.jar,/usr/local/spark-3.5.0-bin-hadoop3/jars/bson-4.11.1.jar pyspark-shell


In [4]:
!pip install pymongo

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 3.4 MB/s eta 0:00:00a 0:00:010m
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 331.1/331.1 kB 13.5 MB/s eta 0:00:00
